# Notebook 02 — PyTorch CV Fundamentals

In Notebook 01 we learned that an image is just a tensor. Now we'll put those
tensors to work: we'll wrap Fashion-MNIST in a PyTorch `Dataset`, transform it
with `torchvision.transforms.v2`, define a tiny CNN from scratch, and do a
single training step by hand -- then a brief 1-epoch training run.

## Learning goals
- Understand the `Dataset` / `DataLoader` abstraction: `__len__`, `__getitem__`,
  batching, shuffling, workers.
- Use modern `torchvision.transforms.v2` pipelines.
- Build a simple CNN (3 conv blocks + 2 FC layers) and count its parameters.
- Run one forward/backward/step cycle manually and watch the loss drop.
- Train for one epoch on a subset and plot the loss curve.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/numberonewastefellow/my_learning/blob/main/notebooks/02_pytorch_cv_fundamentals.ipynb)

In [ ]:
# Universal setup: runs identically in Colab and locally
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("my_learning"):
        !git clone --quiet https://github.com/numberonewastefellow/my_learning.git
    %cd my_learning
    !pip install -q -r requirements.txt

repo_root = os.path.abspath(".") if IN_COLAB else os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils.env import bootstrap
info = bootstrap()
device = info.device


## 1. Dataset and DataLoader

PyTorch's data pipeline is built around two abstractions:

- **`Dataset`** -- a single object that knows *how many* items it contains
  (`__len__`) and *how to produce* item `i` (`__getitem__(i)`).
  Anything that satisfies those two methods is a valid dataset. That flexibility
  is why PyTorch works equally well with images in folders, CSV rows, audio
  clips, or arbitrary tensors.

- **`DataLoader`** -- wraps a `Dataset` and produces **batches**. It handles:
  shuffling between epochs, batching, parallel loading with worker processes,
  and pinning memory for faster host->GPU transfer.

We'll use `torchvision.datasets.FashionMNIST` (a drop-in replacement for
MNIST -- 70k grayscale 28x28 images across 10 clothing categories) as our
source dataset, and wrap it with our own custom `Dataset` so you see the
machinery up close.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import FashionMNIST
from utils.env import data_dir

DATA_DIR = data_dir()

# Underlying torchvision dataset -- we'll download once and reuse.
train_raw = FashionMNIST(root=DATA_DIR, train=True,  download=True)
test_raw  = FashionMNIST(root=DATA_DIR, train=False, download=True)
print("Train size:", len(train_raw), "  Test size:", len(test_raw))
print("Classes:  ", train_raw.classes)

# Inspect one element: a (PIL.Image, int) pair.
img, lbl = train_raw[0]
print("item 0 image type:", type(img).__name__, "size:", img.size, "mode:", img.mode)
print("item 0 label:", lbl, "->", train_raw.classes[lbl])


### A minimal custom `Dataset`

The class below is intentionally thin: it delegates almost everything to
`FashionMNIST` and simply applies an optional transform. In real projects your
custom dataset would be a bit meatier (reading from disk, parsing annotations,
etc.) but the shape is always the same.


In [ ]:
class FashionWrapper(Dataset):
    """Tiny wrapper around FashionMNIST that applies a transform on the fly."""

    def __init__(self, base, transform=None):
        self.base = base
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, lbl = self.base[idx]          # img is a PIL.Image
        if self.transform is not None:
            img = self.transform(img)
        return img, lbl

# We'll add transforms in the next section. For now, use the default (None).
tmp_ds = FashionWrapper(train_raw, transform=None)
print("len(tmp_ds) =", len(tmp_ds))
sample_img, sample_lbl = tmp_ds[7]
print("tmp_ds[7] -> (PIL image of size", sample_img.size, ", label", sample_lbl, ")")


## 2. `torchvision.transforms.v2` -- the modern pipeline

`torchvision` has two transform APIs:

- The **v1** API (`torchvision.transforms`) operates primarily on PIL images.
- The **v2** API (`torchvision.transforms.v2`) operates on **tensors**,
  runs faster (especially on GPU), and composes cleanly with bounding boxes /
  masks when you later do detection or segmentation.

We'll use v2 throughout the course. The standard preprocessing pipeline looks
like this:

1. `v2.ToImage()` -- convert a PIL.Image (or numpy array) to a `tv_tensors.Image`
   tensor of dtype `uint8`, shape `(C, H, W)`.
2. `v2.ToDtype(torch.float32, scale=True)` -- cast to float and rescale
   `uint8` 0..255 into `float` 0..1.
3. `v2.Normalize(mean, std)` -- per-channel mean/std normalization.

For Fashion-MNIST (1 channel) the dataset statistics are
`mean = 0.2860, std = 0.3530` (computed by the torchvision maintainers).


In [ ]:
from torchvision.transforms import v2

MEAN = (0.2860,)
STD  = (0.3530,)

train_tfms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])

train_ds = FashionWrapper(train_raw, transform=train_tfms)
test_ds  = FashionWrapper(test_raw,  transform=train_tfms)

# Peek at a transformed sample.
x, y = train_ds[0]
print("After transform:")
print("  x shape:", tuple(x.shape), " dtype:", x.dtype)
print("  x min/max:", x.min().item(), x.max().item())
print("  y (class idx):", y, "->", train_raw.classes[y])


### Visualise before/after the transform

To see what normalization does, we'll display a few raw images alongside their
denormalized versions (which should look identical to the eye -- the values
differ but after undoing normalization we recover the same picture).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def denorm(t, mean=MEAN, std=STD):
    m = torch.as_tensor(mean).view(-1, 1, 1)
    s = torch.as_tensor(std).view(-1, 1, 1)
    return (t * s + m).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    raw_img, lbl = train_raw[i]             # PIL image, uint8
    x, _        = train_ds[i]               # normalized tensor
    axes[0, i].imshow(np.array(raw_img), cmap="gray")
    axes[0, i].set_title(f"raw: {train_raw.classes[lbl]}", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(denorm(x)[0].numpy(), cmap="gray")
    axes[1, i].set_title("denorm(transform)", fontsize=9)
    axes[1, i].axis("off")
fig.tight_layout();


### Now the DataLoader

A `DataLoader` batches examples and optionally uses worker processes to load
them in parallel. A few kwargs worth knowing:

- **`batch_size`** -- how many samples per mini-batch. 64 is a fine starting
  point for small datasets.
- **`shuffle=True`** -- reshuffle indices at the start of every epoch. Essential
  for training, irrelevant for evaluation.
- **`num_workers`** -- number of background processes. `0` means "do everything
  in the main process" -- safest on Windows/Colab when starting out. Bump this
  up once your code works.
- **`pin_memory=True`** -- allocates a page-locked CPU buffer which makes
  host->GPU transfers noticeably faster. Set it to `True` if you have a GPU.


In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0, pin_memory=(device.type == "cuda"))
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=0, pin_memory=(device.type == "cuda"))

# Peek at one batch.
xb, yb = next(iter(train_loader))
print("xb.shape =", tuple(xb.shape), " xb.dtype =", xb.dtype)
print("yb.shape =", tuple(yb.shape), " yb.dtype =", yb.dtype)
print("first 8 labels:", yb[:8].tolist())


## 3. A tiny CNN from scratch

Time to build our first convolutional network. Here's the architecture:

```
Input:  (N, 1, 28, 28)     # batch of grayscale 28x28 images

Block 1: Conv(1 -> 16, k=3, p=1) -> ReLU -> MaxPool(2)     # -> (N, 16, 14, 14)
Block 2: Conv(16 -> 32, k=3, p=1) -> ReLU -> MaxPool(2)    # -> (N, 32,  7,  7)
Block 3: Conv(32 -> 64, k=3, p=1) -> ReLU -> MaxPool(2)    # -> (N, 64,  3,  3) (28 -> 14 -> 7 -> 3 because 7 // 2 = 3)

Flatten -> FC(64*3*3 -> 128) -> ReLU -> FC(128 -> 10)      # 10 classes
```

A few notes:

- **`padding=1`** keeps the spatial size unchanged across a 3x3 conv; only the
  `MaxPool2d(2)` halves it.
- **`ReLU`** is our non-linearity. Without it, stacking convs collapses to a
  single linear map.
- **`CrossEntropyLoss`** expects *raw logits* (no softmax) plus integer class
  targets -- it fuses log-softmax and NLL for numerical stability.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class TinyCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(1,  16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        # After three pools: 28 -> 14 -> 7 -> 3 (floor div), channels=64.
        self.fc1 = nn.Linear(64 * 3 * 3, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # (N, 16, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))   # (N, 32,  7,  7)
        x = self.pool(F.relu(self.conv3(x)))   # (N, 64,  3,  3)
        x = x.flatten(1)                       # (N, 576)
        x = F.relu(self.fc1(x))                # (N, 128)
        return self.fc2(x)                     # (N, num_classes), raw logits

model = TinyCNN(num_classes=10).to(device)
print(model)

n_params = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {n_params:,}")
print(f"Trainable parameters: {n_train:,}")


In [ ]:
# Sanity check: run one batch through the model and inspect the shape.
model.eval()
with torch.no_grad():
    out = model(xb.to(device))
print("output shape:", tuple(out.shape), "  (batch, num_classes)")
print("first row (raw logits):", out[0].cpu().tolist())


## 4. A single training step, spelled out

Before hiding everything behind a helper, let's perform **one** gradient update
step manually so every piece is visible.

The standard recipe is:

1. `optimizer.zero_grad()` -- clear out leftover gradients from the previous
   step (PyTorch accumulates by default).
2. `logits = model(x)` -- forward pass.
3. `loss = loss_fn(logits, y)` -- scalar loss tensor.
4. `loss.backward()` -- autograd computes `d loss / d parameter` for every
   parameter.
5. `optimizer.step()` -- subtract `lr * gradient` from each parameter (for
   plain SGD; Adam and friends do something fancier but the API is the same).

We expect the loss to drop after one step, at least a little.


In [ ]:
from torch import optim

model.train()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

xb_dev = xb.to(device)
yb_dev = yb.to(device)

# Loss BEFORE the update
with torch.no_grad():
    loss_before = loss_fn(model(xb_dev), yb_dev).item()
print(f"loss before step: {loss_before:.4f}")

# One manual training step
optimizer.zero_grad()
logits = model(xb_dev)
loss   = loss_fn(logits, yb_dev)
loss.backward()
optimizer.step()

# Loss AFTER the update on the SAME batch (should have decreased)
with torch.no_grad():
    loss_after = loss_fn(model(xb_dev), yb_dev).item()
print(f"loss after step:  {loss_after:.4f}   (delta = {loss_after - loss_before:+.4f})")


## 5. A 1-epoch mini-training on a subset

We'll take the first 2,000 training samples, wrap them in a tiny DataLoader,
and train for one full pass. We'll record the loss per batch and plot it so
you can literally see the network learning.

We are deliberately **not** using `utils.training.fit` here -- that gets
introduced in Notebook 03. Keeping it manual reinforces the mechanics from the
previous section.


In [ ]:
from torch.utils.data import Subset

subset_ds = Subset(train_ds, indices=list(range(2000)))
subset_loader = DataLoader(subset_ds, batch_size=64, shuffle=True, num_workers=0,
                           pin_memory=(device.type == "cuda"))
print("subset size:", len(subset_ds), " batches:", len(subset_loader))

# Fresh model + optimizer so this cell is reproducible on its own.
model = TinyCNN(num_classes=10).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

model.train()
batch_losses = []
for step, (xb, yb) in enumerate(subset_loader):
    xb = xb.to(device)
    yb = yb.to(device)

    optimizer.zero_grad()
    logits = model(xb)
    loss   = loss_fn(logits, yb)
    loss.backward()
    optimizer.step()

    batch_losses.append(loss.item())
    if step % 5 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")

print(f"\nfinal batch loss: {batch_losses[-1]:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(batch_losses, marker="o", markersize=3)
plt.xlabel("batch index")
plt.ylabel("CrossEntropy loss")
plt.title("TinyCNN loss over one epoch on a 2000-sample subset")
plt.grid(True, alpha=0.3);


Even on this tiny subset, in a single epoch, the loss drops from roughly 2.3
(random chance across 10 classes is `ln(10) approx 2.30`) to well below 1.
That's a lot of learning from a very small CNN.


## Key Takeaways

- A PyTorch `Dataset` is defined by `__len__` + `__getitem__`. A `DataLoader`
  adds batching, shuffling and parallel loading on top.
- Prefer `torchvision.transforms.v2` -- it's faster and operates on tensors
  natively. The standard pipeline is `ToImage -> ToDtype(float32, scale=True)
  -> Normalize(mean, std)`.
- A CNN alternates `Conv2d -> non-linearity -> pool/stride` blocks and ends in
  fully connected classifier layers. Padding controls spatial size;
  `MaxPool2d(2)` halves it.
- One training step = `zero_grad -> forward -> loss -> backward -> step`.
  Forgetting `zero_grad()` is the classic beginner bug (gradients silently
  accumulate).
- `CrossEntropyLoss` takes raw logits, not softmax probabilities.


## Exercises

1. **Swap the optimizer.** Replace `Adam` with plain `SGD(lr=1e-2, momentum=0.9)`.
   Re-run the manual step. Does the loss still go down? What happens with
   `lr=1.0`?
2. **Increase model capacity.** Add a fourth conv block (64 -> 128 channels)
   and adjust the flatten size. Re-count parameters and re-train on the same
   2000-sample subset. Does the final-batch loss get lower?
3. **Add a data augmentation**. Insert `v2.RandomHorizontalFlip(p=0.5)` **before**
   `Normalize` in `train_tfms`. Does training slow down? Does it change the
   loss curve shape? (Augmentations typically make loss noisier but help
   generalization -- we'll see this clearly in Notebook 03.)
